# Merging and Joining

## Overview

Combining DataFrames by matching on shared keys. Pandas offers two interfaces: `pd.merge()` (SQL-style, column-based) and `pd.concat()` (stacking by axis).

**Join types:**

| Join | Keeps | Use when |
|---|---|---|
| `inner` | Rows with matching keys in both | Only want complete matches |
| `left` | All left rows; NaN for non-matches on right | Left is the primary table |
| `right` | All right rows; NaN for non-matches on left | Right is the primary table |
| `outer` | All rows from both; NaN for non-matches | Need to audit what's missing |

**Rule of thumb:** always check the shape before and after a merge. Unexpected row counts indicate duplicate keys or mismatched key values.

---

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Table 1: site characteristics
sites = pd.DataFrame({
    'site_id':   [f'SITE_{i:03d}' for i in range(1, 51)],
    'catchment': rng.choice(['North', 'South', 'East', 'West'], 50),
    'elevation': rng.uniform(50, 400, 50).round(1),
    'treatment': rng.choice(['control', 'restored'], 50),
})

# Table 2: water quality measurements (not all sites sampled)
sampled_ids = rng.choice(sites['site_id'], size=40, replace=False)
water = pd.DataFrame({
    'site_id':   sampled_ids,
    'nitrate':   rng.gamma(2, 2, 40).round(2),
    'phosphorus':rng.gamma(1.5, 1.5, 40).round(2),
    'ph':        rng.uniform(6.5, 8.5, 40).round(2),
})

# Table 3: species richness (some sites appear twice — resurvey)
bio_ids = list(rng.choice(sites['site_id'], size=35, replace=False)) + \
          list(rng.choice(sites['site_id'][:10], size=5, replace=False))
bio = pd.DataFrame({
    'site_id': bio_ids,
    'survey':  ['primary']*35 + ['resurvey']*5,
    'richness':rng.integers(5, 35, 40),
})

print(f'sites: {sites.shape}, water: {water.shape}, bio: {bio.shape}')

sites: (50, 4), water: (40, 4), bio: (40, 3)


---
## Basic Joins with `pd.merge()`

In [2]:
# Inner join: only sites with water quality data
inner = pd.merge(sites, water, on='site_id', how='inner')
print(f'Inner join: {inner.shape}  (expected ~40 rows)')

# Left join: all sites; NaN where water quality not measured
left = pd.merge(sites, water, on='site_id', how='left')
print(f'Left join:  {left.shape}  (all 50 sites)')
print(f'Sites without water data: {left["nitrate"].isnull().sum()}')

# Outer join: audit coverage across both tables
outer = pd.merge(sites, water, on='site_id', how='outer', indicator=True)
print('\nMerge coverage:')
print(outer['_merge'].value_counts())

Inner join: (40, 7)  (expected ~40 rows)
Left join:  (50, 7)  (all 50 sites)
Sites without water data: 10

Merge coverage:
_merge
both          40
left_only     10
right_only     0
Name: count, dtype: int64


---
## Many-to-One and Many-to-Many Joins

In [3]:
# bio has duplicate site_ids (resurveys) — many-to-one with sites
before = len(sites)
merged_bio = pd.merge(sites, bio, on='site_id', how='left')
print(f'Before: {before} rows → After left join with bio: {len(merged_bio)} rows')
print('Row increase due to resurveyed sites appearing twice')

# Always validate: was the row increase expected?
dupe_sites = bio[bio.duplicated('site_id', keep=False)]['site_id'].unique()
print(f'Sites with multiple bio records: {len(dupe_sites)}')

# If you only want primary surveys:
bio_primary = bio.query('survey == "primary"')
merged_primary = pd.merge(sites, bio_primary, on='site_id', how='left')
print(f'With primary surveys only: {merged_primary.shape}  (back to 50)')

Before: 50 rows → After left join with bio: 54 rows
Row increase due to resurveyed sites appearing twice
Sites with multiple bio records: 4
With primary surveys only: (50, 6)  (back to 50)


---
## Stacking DataFrames with `pd.concat()`

In [4]:
# concat axis=0: stack rows (equivalent to rbind in R)
# Use when two tables have the same columns from different time periods / regions
sites_2022 = sites.iloc[:25].copy().assign(year=2022)
sites_2023 = sites.iloc[25:].copy().assign(year=2023)

combined = pd.concat([sites_2022, sites_2023], axis=0, ignore_index=True)
print(f'Combined: {combined.shape}')

# ignore_index=True resets to clean 0-based index
# Without it, original indices are preserved — can produce duplicate index values

# concat axis=1: stack columns (equivalent to cbind)
# Only safe when rows are guaranteed aligned
site_info = sites[['site_id', 'catchment']].reset_index(drop=True)
nutrient_data = water[['nitrate', 'phosphorus']].reset_index(drop=True)
# Note: this is only valid if both tables are already in the same row order
# Prefer merge() when in doubt

# concat with keys: track source
combined_keyed = pd.concat(
    {'2022': sites_2022, '2023': sites_2023},
    axis=0
).reset_index(level=0).rename(columns={'level_0': 'source_year'})
print(combined_keyed.head(3))

Combined: (50, 5)
  source_year   site_id catchment  elevation treatment  year
0        2022  SITE_001     North      118.1  restored  2022
1        2022  SITE_002      West      213.4   control  2022
2        2022  SITE_003      East       65.3  restored  2022


---
## Validating Merges

In [5]:
# validate= parameter catches unexpected key relationships
try:
    # '1:1' raises if either side has duplicate keys
    pd.merge(sites, water, on='site_id', how='inner', validate='1:1')
    print('1:1 merge validated successfully')
except pd.errors.MergeError as e:
    print(f'MergeError: {e}')

# indicator=True: audit coverage
audit = pd.merge(sites, water, on='site_id', how='outer', indicator=True)
print('\nMerge audit:')
print(audit['_merge'].value_counts())
print('\nSites missing water data:')
print(audit.loc[audit['_merge']=='left_only', 'site_id'].values)

1:1 merge validated successfully

Merge audit:
_merge
both          40
left_only     10
right_only     0
Name: count, dtype: int64

Sites missing water data:
<StringArray>
['SITE_002', 'SITE_009', 'SITE_014', 'SITE_017', 'SITE_027', 'SITE_031',
 'SITE_035', 'SITE_038', 'SITE_042', 'SITE_043']
Length: 10, dtype: str


---

## Common Pitfalls

**1. Not checking row counts before and after a merge**  
A merge that silently produces more rows than either input table indicates duplicate keys — a many-to-many join where a one-to-many was expected. Always print shapes before and after. Use `validate='1:1'` or `validate='1:m'` to make the expected cardinality explicit and raise an error if violated.

**2. Using `concat(axis=1)` instead of `merge()` when rows are not guaranteed aligned**  
`pd.concat([df1, df2], axis=1)` aligns on the index, not on a key column. If both DataFrames have a default 0-based integer index and happen to be the same length, it silently produces a plausible-looking result that may be completely wrong. Use `pd.merge()` on an explicit key column whenever row correspondence is not certain.

**3. Forgetting `ignore_index=True` after `concat(axis=0)`**  
Without `ignore_index=True`, concatenating two DataFrames preserves original indices, producing duplicate index values (0, 1, 2 from both tables). Downstream `.loc[]` access on a duplicated index returns multiple rows rather than a single row.

**4. Merging on columns with different dtypes**  
Merging on `'site_id'` when one table has it as `int64` and the other as `object` (string) produces an empty result — no keys match because `1 != '1'`. Always verify that join key columns have identical dtypes before merging.

**5. Not using `indicator=True` to audit coverage**  
A left join that drops many right-side matches is silent without `indicator=True`. Always use `indicator=True` on the first merge of a new dataset to verify that key coverage is as expected, then remove the indicator column before proceeding.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*